#### __libraries__

In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

In [7]:
from shared_scripts.statistical_analysis import t_margin_of_error, permutation_test_median

In [8]:
from shared_scripts.bb2_data_load import BB2_load_into_dataframe

In [54]:
from shared_scripts.additional_functions import extract_significance_symbols

#### __plt params__

In [10]:
plt.rcParams["legend.frameon"] = False
plt.rcParams['legend.fontsize'] = 12
plt.rcParams['axes.labelsize'] = 19
plt.rcParams['axes.labelweight'] = 'bold'
plt.rcParams['xtick.labelsize'] = 16
plt.rcParams['ytick.labelsize'] = 16  
plt.rcParams['font.size'] = 12
plt.rcParams['figure.dpi'] = 1000
plt.rcParams['font.family'] = 'Calibri'

#### __func__

In [16]:
def _2C_data_process(data, **kwargs):
    
    #chck the input data
    if not isinstance(data, pd.DataFrame) or data.empty:
        raise ValueError("input must be a non-empty pandas DataFrame")
        
    if not all(pd.api.types.is_numeric_dtype(dtype) for dtype in data.dtypes):
        raise TypeError("All columns in inout must be numerical")
        
    #assign timepoint
    frq= kwargs.get('frequency', np.NaN)
    indel= kwargs.get('initial_delay', np.NaN)
    if "frequency" not in kwargs or "initial_delay" not in kwargs:
        raise ValueError("frequency and initial_delay must be provided")
    try:
        data= data.assign(timepoint_minutes=data['frame'] * frq + indel).drop(columns=["frame"])
    except Exception as ex:
        raise ValueError("failed to calculate the timepoints in minutes")
        
    #set the index 
    if 'timepoint_minutes' in data.columns:
        data= data.set_index('timepoint_minutes')
    else:
        raise ValueError("dataset is missing a frame column")
        
    #calculate the mean
    try:
        data= data.assign(avg_distance_ctrl= data.iloc[:, 0:3].mean(axis= 1),
                          avg_distance_lata= data.iloc[:, 3:6].mean(axis= 1))
    except Exception as ex:
        print(f'failed to calculate the mean: {ex}')
        data= data.assign(avg_distance_ctrl= np.NaN,
                          avg_distance_lata= np.NaN)
        
    #calculate the MOE95
    try:
        data= data.assign(moe95_ctrl= data.apply(lambda x: t_margin_of_error(x.iloc[0:3]), axis= 1),
                          moe95_lata= data.apply(lambda x: t_margin_of_error(x.iloc[3:6]), axis= 1))
    except Exception as ex:
        print(f'failed to calculate the moe95: {ex}')
        data= data.assign(moe95_ctrl= np.NaN,
                          moe95_lata= np.NaN)
        
    return data

#### __data load__

In [19]:
_6B= BB2_load_into_dataframe('Figure_6B.csv')

In [20]:
_6C= BB2_load_into_dataframe('Figure_6C.csv')

#### __data processing__

In [22]:
_6C_final= _2C_data_process(_6C, frequency= 6, initial_delay= 12)

#### __permutation test__

In [42]:
# permutation_test_median(_6B, category_column= 'group',  reference_group= 'dmso_ctrl', tested_parameter= 'alpha', n_resamples= 100000, export= True, path= r"...\Fig_6B_alpha_permutation_test.csv")

In [44]:
# permutation_test_median(_6B, category_column= 'group', reference_group= 'dmso_ctrl', tested_parameter= 'K', n_resamples= 100000, export= True, path= r"...\Fig_6B_K_permutation_test.csv")

#### __significance symbols__

In [62]:
significance_alpha= extract_significance_symbols(path= r"C:\Users\Jakub\Desktop\figure_scripts\permutation_test\Fig_6B_alpha_permutation_test.csv")
significance_k= extract_significance_symbols(path= r"C:\Users\Jakub\Desktop\figure_scripts\permutation_test\Fig_6B_K_permutation_test.csv")

#### __visuals__

In [27]:
def fig6(_6B_data, _6C_data, **kwargs):
    
    #fig + gridspec
    fig= plt.figure(figsize= (12.5, 20), constrained_layout= True)
    gs= gridspec.GridSpec(8, 6, figure=fig)

    ax1= fig.add_subplot(gs[0:2, 0:3])
    ax2= fig.add_subplot(gs[0:2, 3:6])
    ax3= fig.add_subplot(gs[2:4, 0:6])
    ax4= fig.add_subplot(gs[4:6, 0:3])
    ax5= fig.add_subplot(gs[4:6, 3:6])
    ax6= fig.add_subplot(gs[6:7, 0:3])
    ax7= fig.add_subplot(gs[6:7, 3:6])
    ax8= fig.add_subplot(gs[7:8, 0:2])
    ax9= fig.add_subplot(gs[7:8, 2:4])
    ax10= fig.add_subplot(gs[7:8, 4:6])
    
    ###fig 6B
    #alpha
    box_position = 1.5 
    points_bottom, points_top= 0.25, 0.75 
    colors= plt.cm.tab10(np.linspace(0, 1, 10))
    for group in _6B_data.group.unique():
        dark= '#00527C' if group == 'dmso_ctrl' else '#FF781F'
        light= '#DFE9F5' if group == 'dmso_ctrl' else '#FEE7CC'
        data= _6B_data.loc[_6B_data.group== group]
        ax1.boxplot(data.alpha, 
                      positions=[box_position], 
                      widths=0.5,
                      showfliers=False,  
                      patch_artist= True,
                      boxprops=dict(facecolor=light, edgecolor=dark, linewidth=1.5),
                      whiskerprops=dict(color=dark, linewidth=1.5),
                      capprops=dict(color=dark, linewidth=1.5),
                      medianprops=dict(color=dark, linewidth=2))
        symbols_alpha= kwargs.get('significance_symbols_alpha', None)
        if symbols_alpha:
            significance_symbol_position_y= np.max(iqr_outlier_removal(data.alpha))
            ax1.annotate(symbols_alpha.get(strain, ''), (box_position, significance_symbol_position_y), fontweight= 'bold', textcoords= 'offset points', xytext= (0, 10), ha= 'center')
        x_points = np.random.uniform(points_bottom, points_top, len(data.alpha))
        ax1.scatter(x_points,
                      data.alpha,
                      alpha=0.66,
                      color=light,
                      edgecolors="white",
                      linewidths=0.2,
                      s=12,
                      zorder=3)
        #K
        ax2.boxplot(data.K, 
                      positions=[box_position], 
                      widths=0.5,
                      showfliers=False,  
                      patch_artist= True,
                      boxprops=dict(facecolor=light, edgecolor=dark, linewidth=1.5),
                      whiskerprops=dict(color=dark, linewidth=1.5),
                      capprops=dict(color=dark, linewidth=1.5),
                      medianprops=dict(color=dark, linewidth=2))
        symbols_k= kwargs.get('significance_symbols_k', None)
        if symbols_k:
            significance_symbol_position_y= np.max(iqr_outlier_removal(data.K))
            ax2.annotate(symbols_k.get(strain, ''), (box_position, significance_symbol_position_y), fontweight= 'bold', textcoords= 'offset points', xytext= (0, 10), ha= 'center')
        ax2.scatter(x_points,
                      data.K,
                      alpha=0.66,
                      color=light,
                      edgecolors="white",
                      linewidths=0.2,
                      s=12,
                      zorder=3)
        box_position += 2
        points_bottom += 2
        points_top += 2
    #rename the groups
    _6B_data= _6B_data.assign(group= np.where(_6B_data.group=='dmso_ctrl', 'As(III)', 'As(III) + LatA'))
    ax1.set_ylabel('alpha', weight= 'bold')
    ax1.set_ylim(bottom=-0.55, top= 2.55)
    ax1.axhline(_6B_data.loc[_6B_data.group=='As(III)', 'alpha'].median(), ls= '--', lw= .66, color="#00527C", alpha= .66, zorder= 5)
    ax1.set_xlim(0, len(_6B_data.group.unique())*2)
    ax1.set_xticks(np.arange(1, len(_6B_data.group.unique())*2, 2))
    ax1.set_xticklabels(_6B_data.group.unique())
    ax1.set_yticks([-0.5, 0.5, 1.5, 2.5])
    ax2.set_ylabel('K', weight= 'bold')
    ax2.set_ylim(bottom=-2, top= 122)
    ax2.axhline(_6B_data.loc[_6B_data.group=='As(III)', 'K'].median(), ls= '--', lw= .66, color="#00527C", alpha= .66, zorder= 5)
    ax2.set_xlim(0, len(_6B_data.group.unique())*2)
    ax2.set_xticks(np.arange(1, len(_6B_data.group.unique())*2, 2))
    ax2.set_xticklabels(_6B_data.group.unique())
    ax2.set_yticks([0, 40, 80, 120])
    
    ###fig 6C
    ax3.errorbar(_6C_data.index, _6C_data.avg_distance_ctrl, lw= 2.5, label= 'As(III)', color= '#00527C')
    y_min_ctrl= _6C_data.avg_distance_ctrl - _6C_data.moe95_ctrl
    y_max_ctrl= _6C_data.avg_distance_ctrl + _6C_data.moe95_ctrl
    ax3.fill_between(_6C_data.index,  y_min_ctrl,  y_max_ctrl,  color='#DFE9F5',  label="margin of error area: As(III)", alpha= .66)
    ax3.errorbar(_6C_data.index, _6C_data.avg_distance_lata, lw= 2.5, label= 'As(III) + LatA', color= '#FF781F')
    y_min_ctrl= _6C_data.avg_distance_lata - _6C_data.moe95_lata
    y_max_ctrl= _6C_data.avg_distance_lata + _6C_data.moe95_lata
    ax3.fill_between(_6C_data.index,  y_min_ctrl,  y_max_ctrl,  color='#FEE7CC',  label="margin of error area: As(III) + LatA", alpha= .66)
    ax3.set_xlabel('time (min)', weight= 'bold')
    ax3.set_ylabel('average movement', weight= 'bold') #pxls/frame
    ax3.set_ylim(0, 8)
    ax3.set_yticks([1, 3, 5, 7])
    ax3.legend(ncol= 2)
    ax3.set_xticks([0,60,120,180,240,300,360])

In [29]:
fig6(_6B, _6C_final)